# Known-repeater controls and DAS-only adjudication registration

This checkpoint makes the next scientific test explicit. The development DAS run recovered 2/2 known local catalog events, but that is only a sanity check, not a family-recall estimate. The held-out population contains 21 DAS-only rows that now require independent adjudication. This notebook reads compact registered products only; it opens no raw HDF5, station waveform, new catalog-event, or family-assignment rows.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config' / 'heldout_das_adjudication.json').is_file())
CONFIG_PATH = ROOT / 'config' / 'heldout_das_adjudication.json'
STATUS_PATH = ROOT / 'outputs' / 'heldout_v2' / 'registration' / 'adjudication_registration_status.json'
CONFIG = json.loads(CONFIG_PATH.read_text())
STATUS = json.loads(STATUS_PATH.read_text())
sha256 = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha256(CONFIG_PATH) == STATUS['adjudication_config_sha256']
assert STATUS['status'] == 'PASS'
assert STATUS['heldout_DAS_only_candidate_count'] == 21
assert STATUS['known_local_development_positive_control_recovery'] == '2/2'
assert STATUS['family_assignments_made'] == 0
print('PASS: adjudication registration and access ledger verify')

In [ ]:
context = pd.read_csv(ROOT / 'outputs' / 'heldout_v2' / 'comparison' / 'das_network_network_context.csv')
das_only = context[context['comparison_membership'] == 'DAS_only'].copy()
assert len(das_only) == 21
assert das_only['DAS_independent_adjudication_status'].eq('pending_independent_catalog_forced_network_score_waveform_and_DAS_artifact_review').all()
assert das_only['repeater_family_assignment'].eq('not_assigned').all()
display(das_only[['comparison_candidate_id', 'interval_id', 'DAS_candidate_id', 'DAS_trigger_time', 'DAS_coincidence_score', 'DAS_strong_block_count']])

In [ ]:
controls = pd.DataFrame(CONFIG['known_positive_controls']['controls'])
display(controls)
assert len(controls) == 2
assert sorted(controls['DAS_score_rank'].tolist()) == [1, 2]
assert sorted(controls['strong_block_count_of_10'].tolist()) == [8, 10]
print('These controls establish detector capability on two development events, not repeater-family recall.')

In [ ]:
population = CONFIG['published_repeater_population']
summary = pd.DataFrame({
    'quantity': ['published-family events', 'published families', 'post-2014 continuations', 'heldout DAS-only rows'],
    'value': [population['waldhauser_schaff_event_count'], population['waldhauser_schaff_family_count'], population['post_2014_continuation_count'], STATUS['heldout_DAS_only_candidate_count']]
})
display(summary)
print('Family partition remains STOP because the independent exact-ID catalogs disagree.')

## Next scientific operation

Run independent adjudication of all 21 DAS-only rows: force the frozen network detector at each DAS time, check registered catalog/regional associations, review DAS persistence and channel/block morphology, inspect available waveforms, and report interval-stratified uncertainty. No threshold repair, candidate deletion, matching-window sweep, or family assignment is allowed.

A positive result requires at least one independently validated local DAS-only event beyond the full network union. A family-classification claim additionally requires reconciling the M00413/M00414 partition conflict.